[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/05_03_main_embeddings.ipynb)

# Module 5, NLP: Discovering Structure With Embeddings

**Notebook:** `05_03_main_embeddings`

## What we're doing

Suppose we have a large collection of text and **no labels telling us what the categories are**.

In the previous NLP examples, we began with a known outcome: default versus non-default, or Hamilton versus Madison. Here the problem is different:

> **Can we discover meaningful structure in a corpus when nobody has labeled it for us?**

We will use recent machine-learning research abstracts as our corpus. The goal is not to produce a definitive taxonomy of machine learning. The goal is to see how an unsupervised NLP pipeline can help us explore a body of text we do not yet understand.

By the end of the notebook, we will have:

1. collected text from a live API;
2. encoded every abstract as a semantic vector;
3. clustered nearby vectors;
4. interpreted the clusters using distinctive vocabulary;
5. projected the high-dimensional space into two dimensions;
6. inspected representative papers to see whether our interpretation holds up.

## The recipe

| Step | Tool | What it does |
| --- | --- | --- |
| Acquire | OpenAlex API | retrieve a corpus |
| Encode | Sentence Transformers | turn each abstract into a semantic vector |
| Cluster | KMeans | group nearby vectors |
| Interpret | cluster-level TF-IDF | surface distinctive terms |
| Project | UMAP | create a 2-D visualization |
| Verify | representative papers | check whether our labels make sense |

### One important distinction

This workflow is **inspectable**, but not every component is directly interpretable.

A Flesch-Kincaid score has a human-readable meaning. A logistic-regression coefficient can be tied to a named feature. An embedding dimension usually cannot. What we *can* inspect is the behavior of the overall system: which texts end up near one another, which clusters appear, which terms distinguish them, and which documents sit near each cluster center.


## 0) Setup

We use two non-default packages:

- `sentence-transformers` for semantic embeddings;
- `umap-learn` for the 2-D projection.

In Google Colab, the first run may need to download the libraries and embedding model.

In [ ]:
import os
import re
import time
import textwrap

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    import umap
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers", "umap-learn"
    ])

    from sentence_transformers import SentenceTransformer
    import umap

print("Setup complete.")

## 1) Acquire a corpus from OpenAlex

For the walkthrough, we will retrieve the corpus **live** from the OpenAlex API.

This is useful because it exposes an important part of real NLP work that is easy to hide in classroom examples:

> Before you can analyze text, you have to obtain it.

Our query asks for machine-learning papers published from 2023 onward that have abstracts available.

### A sampling caveat

We sort the results by citation count and stop after a fixed number of papers. That makes the walkthrough manageable, but it also means this corpus is **not a representative sample of all recent machine-learning research**.

Think of it as:

> a convenient sample of highly cited recent papers returned by this query.

That matters later. If one cluster is larger than another, we can say it occupies a larger share of **this corpus**. We should not automatically conclude that the corresponding subfield produces more research in the world.

In [ ]:
DATA_DIR = "assets/data"
os.makedirs(DATA_DIR, exist_ok=True)

CACHE_PATH = os.path.join(DATA_DIR, "openalex_ml_corpus.csv")

BASE = "https://api.openalex.org/works"

# OpenAlex concept ID used in this course notebook for "Machine learning"
CONCEPT_ID = "C119857082"

MAX_WORKS = 2000
FROM_YEAR = 2023

In [ ]:
def inverted_index_to_text(inv):
    """Reconstruct an abstract stored by OpenAlex as token -> positions."""
    if not isinstance(inv, dict) or len(inv) == 0:
        return None

    max_pos = 0
    for positions in inv.values():
        if positions:
            max_pos = max(max_pos, max(positions))

    tokens = [""] * (max_pos + 1)

    for token, positions in inv.items():
        for p in positions:
            if 0 <= p < len(tokens) and tokens[p] == "":
                tokens[p] = token

    text = " ".join(t for t in tokens if t)
    return text if text.strip() else None


def fetch_ml_corpus(max_works=MAX_WORKS):
    """Fetch a recent machine-learning corpus from OpenAlex."""
    rows = []
    cursor = "*"

    while len(rows) < max_works:
        params = {
            "per-page": 200,
            "cursor": cursor,
            "filter": (
                f"has_abstract:true,"
                f"from_publication_date:{FROM_YEAR}-01-01,"
                f"concept.id:{CONCEPT_ID}"
            ),
            "sort": "cited_by_count:desc",
        }

        response = requests.get(BASE, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()

        for work in payload.get("results", []):
            if len(rows) >= max_works:
                break

            abstract = inverted_index_to_text(
                work.get("abstract_inverted_index")
            )

            if not abstract or len(abstract) < 200:
                continue

            rows.append({
                "openalex_id": work.get("id"),
                "title": work.get("title"),
                "publication_date": work.get("publication_date"),
                "cited_by_count": work.get("cited_by_count", 0),
                "abstract": abstract,
            })

        cursor = payload.get("meta", {}).get("next_cursor")
        if not cursor:
            break

        time.sleep(0.15)

    return pd.DataFrame(rows)

In [ ]:
if os.path.exists(CACHE_PATH):
    df = pd.read_csv(CACHE_PATH)
    print(f"Loaded cached corpus: {len(df):,} papers")
else:
    print("Fetching from OpenAlex...")
    df = fetch_ml_corpus()
    df.to_csv(CACHE_PATH, index=False)
    print(f"Fetched and cached {len(df):,} papers.")

df.head(3)

### Pause and inspect the corpus

Before doing any machine learning, look at a few rows.

Questions to ask:

- What information do we have for each paper?
- What information do we **not** have?
- Do the titles already suggest multiple research themes?
- What biases might be introduced by the way we sampled these papers?

This is an unsupervised problem. We are not starting with a `topic` column that tells us the correct answer.

## 2) Light cleaning

We do very little cleaning.

For embeddings, aggressive preprocessing can actually remove useful context. We will simply remove URLs and normalize whitespace.

In [ ]:
url_pat = re.compile(r"https?://\S+|www\.\S+")
multi_space_pat = re.compile(r"\s+")

def clean_text(s):
    if not isinstance(s, str):
        return ""

    s = url_pat.sub(" ", s.strip())
    return multi_space_pat.sub(" ", s).strip()

df["text"] = df["abstract"].apply(clean_text)

print(
    f"Cleaned {len(df):,} abstracts. "
    f"Average length: {int(df['text'].str.len().mean())} characters."
)

## 3) Encode each abstract as a semantic vector

We use `all-MiniLM-L6-v2`, a compact Sentence Transformer model.

For each abstract, the model produces a **384-dimensional embedding**.

Conceptually:

\[
\text{text} \longrightarrow [x_1, x_2, \ldots, x_{384}]
\]

The important difference from our earlier feature-engineering exercise is that **we did not decide what those 384 dimensions should be**.

With Flesch-Kincaid, we chose a measurable feature and knew what it represented.

With embeddings, the representation is learned by the model. Individual dimensions are not usually meaningful to us, but semantically related texts tend to receive similar vector representations.

That makes embeddings useful for tasks such as:

- semantic search;
- recommendation;
- clustering;
- similarity analysis;
- classification.

### The conceptual jump

Bag-of-words methods mainly capture which terms appear.

Embeddings try to capture **semantic similarity**: texts can be close even when they do not use exactly the same vocabulary.

In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

emb = encoder.encode(
    df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Embedding matrix:", emb.shape)
print("(rows = papers, columns = embedding dimensions)")

### Stop here for a moment

If the output is:

```text
(2000, 384)
```

then we have converted roughly 2,000 pieces of prose into a numerical matrix.

That is the bridge from NLP back to ordinary machine learning.

The remaining methods—clustering and dimensionality reduction—operate on numbers, not directly on sentences.

## 4) Cluster the embedding space with KMeans

If semantically similar papers receive similar vectors, then papers about related ideas should occupy nearby regions of the embedding space.

KMeans asks us to choose the number of clusters, \(K\).

We will begin with:

```python
K = 10
```

This is **not a claim that machine learning has exactly ten true subfields**.

It is an analytical choice that controls the granularity of the solution:

- smaller `K` → broader themes;
- larger `K` → finer themes.

In unsupervised learning, there may not be a single objectively correct answer.

In [ ]:
K = 10

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10,
)

df["cluster"] = kmeans.fit_predict(emb)

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())

### What did KMeans give us?

At this point, we have clusters numbered `0` through `9`.

But a number is not an explanation.

The algorithm has identified groups of nearby vectors. **We** still have to determine what those groups mean.

## 5) Interpret the clusters with cluster-level TF-IDF

To understand a cluster, we can ask:

> Which words and phrases are unusually characteristic of the documents in this group?

We will use a simple **cluster-level TF-IDF** approach:

1. combine all abstracts in a cluster into one large pseudo-document;
2. treat each cluster as one document;
3. calculate TF-IDF across those cluster documents;
4. inspect the highest-weighted terms.

This is conceptually similar to the class-based TF-IDF idea used in topic-modeling systems such as BERTopic, but here we are implementing a straightforward cluster-level TF-IDF calculation ourselves.

The output does **not** automatically name the clusters. It gives us evidence we can use to interpret them.

In [ ]:
# Combine all abstracts within each cluster into one pseudo-document.
cluster_docs = [
    " ".join(df.loc[df["cluster"] == c, "text"])
    for c in range(K)
]

cluster_tfidf = TfidfVectorizer(
    stop_words="english",
    min_df=2,
    ngram_range=(1, 2),
    max_features=20_000,
)

M = cluster_tfidf.fit_transform(cluster_docs)
terms = cluster_tfidf.get_feature_names_out()

TOP_N = 10
cluster_top_terms = {}

for c in range(K):
    row = M[c].toarray().ravel()
    top_idx = row.argsort()[::-1][:TOP_N]
    cluster_top_terms[c] = [terms[i] for i in top_idx]

print("Distinctive terms by cluster:\n")

for c in range(K):
    n = int((df["cluster"] == c).sum())
    top = ", ".join(cluster_top_terms[c])
    print(f"Cluster {c} (n={n:>4d}): {top}")

### Your turn: name the clusters

Before looking at individual papers, try to give each cluster a short human-readable label.

For example, a list containing terms such as:

```text
image, object detection, segmentation, vision
```

might reasonably be interpreted as **computer vision**.

Do not assume your first label is correct. Treat it as a hypothesis that we still need to validate.

Questions to consider:

- Which clusters are easy to name?
- Which seem mixed?
- Do any two clusters appear to cover similar themes?
- What might happen if we increased or decreased `K`?

## 6) Project 384 dimensions into two dimensions with UMAP

Our embeddings live in 384 dimensions. Humans cannot visualize that directly.

UMAP creates a **2-D projection** designed to preserve local neighborhood structure reasonably well.

That gives us something we can plot.

### Important caution

The UMAP map is **not the original embedding space**, and it should not be interpreted like a geographic map.

Use it to see broad neighborhood structure and possible separation. Do not treat exact 2-D distances, blob shapes, or gaps as definitive statistical evidence.

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)

coords = reducer.fit_transform(emb)

print("2-D coordinates:", coords.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

cmap = plt.cm.tab10

for c in range(K):
    mask = (df["cluster"] == c).values
    short_label = ", ".join(cluster_top_terms[c][:2])

    ax.scatter(
        coords[mask, 0],
        coords[mask, 1],
        s=12,
        alpha=0.55,
        color=cmap(c % 10),
        label=f"{c}: {short_label}",
    )

    # Label the approximate 2-D center of each cluster.
    cx = coords[mask, 0].mean()
    cy = coords[mask, 1].mean()

    ax.annotate(
        str(c),
        (cx, cy),
        fontsize=14,
        fontweight="bold",
        ha="center",
        va="center",
        bbox=dict(
            boxstyle="circle,pad=0.3",
            facecolor="white",
            edgecolor="black",
            alpha=0.85,
        ),
    )

ax.set_title(
    f"UMAP projection of {len(df):,} machine-learning papers "
    f"({K} KMeans clusters)"
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
    framealpha=0.9,
)

plt.tight_layout()
plt.show()

### What can we conclude from the map?

Useful observations include:

- some clusters occupy fairly distinct neighborhoods;
- some overlap substantially;
- some clusters may contain several local pockets.

What we should **not** conclude:

- that the 2-D geometry is the true geometry of the corpus;
- that a visible gap proves two categories are fundamentally different;
- that a visually tight cluster is automatically meaningful.

Visualization helps us generate and inspect hypotheses. It does not remove the need to inspect the underlying documents.

## 7) Verify the interpretation using representative papers

Distinctive words can mislead, and a colorful UMAP plot can look convincing even when our labels are poor.

A stronger check is to inspect actual documents.

For each cluster, we will look at the papers whose embeddings are closest to that cluster's centroid.

Then ask:

> Do these papers actually support the label we gave the cluster?

In [ ]:
centroids = kmeans.cluster_centers_

def closest_to_centroid(c, n=5):
    """Return the n papers closest to cluster c's centroid."""

    mask = (df["cluster"] == c).values
    sub_emb = emb[mask]
    sub_idx = np.where(mask)[0]

    centroid_norm = (
        centroids[c] /
        (np.linalg.norm(centroids[c]) + 1e-12)
    )

    similarities = sub_emb @ centroid_norm
    top_local = similarities.argsort()[::-1][:n]

    return df.iloc[sub_idx[top_local]]


for c in range(K):
    label = ", ".join(cluster_top_terms[c][:3])

    print(f"\n=== Cluster {c}: {label} ===")

    reps = closest_to_centroid(c, n=3)

    for _, row in reps.iterrows():
        title = textwrap.shorten(
            str(row["title"]),
            width=110,
            placeholder="…",
        )
        print(f"  • {title}")

### The validation question

For each cluster, compare:

1. its distinctive terms;
2. its location on the UMAP plot;
3. its representative paper titles.

If all three tell roughly the same story, your interpretation becomes more credible.

If they conflict, that is not a failure. It is a signal to investigate:

- perhaps `K` is too small or too large;
- perhaps a cluster contains several related themes;
- perhaps the vocabulary-based label is misleading;
- perhaps the embedding model is grouping documents in a way we did not expect.

## 8) Examine cluster sizes

Cluster size tells us how much of **this sampled corpus** falls into each discovered group.

Because our corpus was filtered and sorted before we collected it, cluster size should not be interpreted as a direct measure of worldwide research activity.

It is still useful descriptively:

> Which themes occupy the largest share of the corpus we analyzed?

In [ ]:
sizes = df["cluster"].value_counts().sort_index()

labels = [
    f"{c}: {', '.join(cluster_top_terms[c][:2])}"
    for c in range(K)
]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    range(K),
    sizes.values,
)

ax.set_yticks(range(K))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Number of papers in this corpus")
ax.set_title("Cluster sizes in the sampled corpus")

for bar, n in zip(bars, sizes.values):
    ax.text(
        bar.get_width() + 5,
        bar.get_y() + bar.get_height() / 2,
        f"{n}",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()


### A Note to Future Instructors Using This Notebook

Hi! It's Joel. If you are using this at some point in the future, you might want to just freeze the data, instead of using an API. I've done that, and it's on the course github page. But it might not be there when you read this. Should be easy enough to run this file, it creates that csv, then use that instead of the API (using that live is a bold choice!) Good luck!

Here is a 

```python
course_corpus = df[
    ["openalex_id", "title", "publication_date",
     "cited_by_count", "abstract"]
].copy()

course_corpus.to_csv("openalex_ml_corpus.csv", index=False)
```

Then place `openalex_ml_corpus.csv` in:

```text
way/nlp_data/
```


In [ ]:
# Stable course-data location -- is it there?
COURSE_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "tunnel-ai/way/main/nlp_data/openalex_ml_corpus.csv"
)

try:
    exercise_df = pd.read_csv(COURSE_DATA_URL)
    print(f"Loaded fixed course corpus: {len(exercise_df):,} papers")
    display(exercise_df.head(3))
except Exception:
    print(
        "The loading code is ready, but the fixed course CSV has not "
        "been found at the GitHub URL yet. Add openalex_ml_corpus.csv "
        "to the repository's nlp_data folder before distributing the exercise."
    )

## 10) What we just did

Starting with raw abstracts and **no topic labels**, we built an unsupervised NLP pipeline:

\[
\text{raw text}
\rightarrow
\text{embeddings}
\rightarrow
\text{clusters}
\rightarrow
\text{human interpretation}
\]

We also added two important validation steps:

- inspect distinctive terms;
- inspect representative documents.

And we used UMAP as a visualization tool rather than treating the picture as proof.

## The larger lesson

Compare this notebook with the earlier NLP examples:

| Problem | Representation | Learning task |
| --- | --- | --- |
| Loan requests | hand-built linguistic + sentiment features | supervised prediction |
| Federalist authorship | stylometric features | supervised classification |
| Research-paper corpus | semantic embeddings | unsupervised discovery |

The question determines the representation and the method.

### Things to experiment with

Try changing one decision at a time:

- `K = 6`, `10`, or `15`;
- UMAP `n_neighbors`;
- UMAP `min_dist`;
- the embedding model;
- the corpus itself.

Then ask the same question each time:

> **Does the new representation produce a more useful interpretation of the corpus, or just a different one?**

That is the central challenge of unsupervised analysis: there may be no single answer key, so interpretation and validation matter.